# Выполнение ЛР №6: "Поиск ассоциативных правил" 

## Подключение библиотек

In [1]:
import os
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori as mlxtend_apriori, association_rules as mlxtend_association_rules

pd.set_option('display.max_rows', None)

## Задание

Реализовать  на  любом языке программирования алгоритм поиска ассоциативных правил: Apriori

Для  проверки  корректности 
алгоритма на любом предлагаемом датасете предлагается 
применить библиотеку mlxtend  языка Python.

## Изучение датасета

In [2]:
# Загрузка датасета Anketa1

path_anketa = os.path.join(os.getcwd(), 'Задание', 'dataset', 'Anketa1.txt')
df_raw = pd.read_csv(path_anketa, sep='\t', decimal=',', encoding='cp1251', encoding_errors='replace')
df_raw.head(10)

,КодАнкеты,Фамилия,Имя,Отчество,"Сумма кредита, руб#",Количество лет проживания в регионе,Социальный статус,Социальный статус супруга(и),Образование,"Стаж работы, лет",Личный доход в месяц после налогооблажения,Наличие личного автомобиля,"Рыночная стоимость автомобиля, руб#",Жилая недвижимость в собственности,"Рыночная стоимость недвижимости, руб#",Наличие кредитов,Возврат кредита
0,3049,Абаджев,Николай,Васильевич,47000,15,Работаю/служу,Работает/служит,среднее,8.5,10000,нет,0,Нет,0,Нет,0
1,4000,Широкова,Светлана,Николаевна,54000,19,Работаю/служу,н/д,начальное,1.0,17000,да,195000,Квартира в многоквартирном доме,1000000,Нет,1
2,3052,Попков,Вячеслав,Леонидович,38000,8,Работаю/служу,н/д,высшее,8.0,10500,да,35000,Квартира в многоквартирном доме,1480000,Нет,0
3,3053,Беляев,Юрий,Алефтинович,25000,19,Работаю/служу,Работает/служит,среднее,1.0,9500,нет,0,Нет,0,Нет,0
4,3055,Репников,Аркадий,Ильич,35000,27,Работаю/служу,Работает/служит,неоконченное высшее,2.0,7500,нет,0,Квартира в многоквартирном доме,900000,Нет,1
5,3056,Калугин,Анатолий,Алексеевич,58000,10,Работаю/служу,Работает/служит,среднее,1.0,9000,нет,0,Квартира в многоквартирном доме,1000000,Да,1
6,3058,Смольникова,Нина,Дмитриевна,44000,6,Работаю/служу,Работает/служит,высшее,1.0,11000,нет,0,Нет,0,Нет,1
7,3060,Катков,Андрей,Викторович,47000,13,Работаю/служу,н/д,среднее,4.0,6000,нет,0,Нет,0,Нет,1
8,3061,Абаев,Александр,Викторович,32000,6,Работаю/служу,Работает/служит,несколько высших или ученая степень,6.0,8000,да,100000,Квартира в многоквартирном доме,880000,Нет,1
9,3062,Кудабаев,Рустам,Альбертович,57000,2,Работаю/служу,н/д,высшее,2.0,17000,нет,0,Квартира в многоквартирном доме,120000,Нет,1


In [3]:
# изучение данных
print('Размер:', df_raw.shape)
print('\nТипы и пропуски:')
print(df_raw.dtypes)

Размер: (83, 17)

Типы и пропуски:
КодАнкеты                                       int64
Фамилия                                        object
Имя                                            object
Отчество                                       object
Сумма кредита, руб#                             int64
Количество лет проживания в регионе             int64
Социальный статус                              object
Социальный статус супруга(и)                   object
Образование                                    object
Стаж работы, лет                              float64
Личный доход в месяц после налогооблажения      int64
Наличие личного автомобиля                     object
Рыночная стоимость автомобиля, руб#             int64
Жилая недвижимость в собственности             object
Рыночная стоимость недвижимости, руб#           int64
Наличие кредитов                               object
Возврат кредита                                 int64
dtype: object


In [4]:
# Уникальные значения категориальных признаков (для бинаризации)
cat_cols = [
    'Социальный статус',
    'Социальный статус супруга(и)',
    'Образование',
    'Наличие личного автомобиля',
    'Жилая недвижимость в собственности',
    'Наличие кредитов'
]
cat_uniques = {col: df_raw[col].astype(str).unique() for col in cat_cols}
df_cat_uniques = pd.DataFrame(dict([(col, pd.Series(vals)) for col, vals in cat_uniques.items()]))
display(df_cat_uniques)

,Социальный статус,Социальный статус супруга(и),Образование,Наличие личного автомобиля,Жилая недвижимость в собственности,Наличие кредитов
0,Работаю/служу,Работает/служит,среднее,нет,Нет,Нет
1,NaN,н/д,начальное,да,Квартира в многоквартирном доме,Да
2,NaN,Не работает,высшее,NaN,Дом,да
3,NaN,Студент,неоконченное высшее,NaN,NaN,NaN
4,NaN,NaN,несколько высших или ученая степень,NaN,NaN,NaN


## Подготовка DataFrame для ассоциативных правил

In [5]:
# Удаляем идентификаторы
cols_drop = ['КодАнкеты', 'Фамилия', 'Имя', 'Отчество']
df = df_raw.drop(columns=[c for c in cols_drop if c in df_raw.columns], errors='ignore')

# Унификация да/нет (приведём к одному регистру)
binary_cols = ['Наличие личного автомобиля', 'Наличие кредитов']
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
        df.loc[df[col].str.contains('да', na=False), col] = 'да'
        df.loc[df[col].str.contains('нет', na=False), col] = 'нет'

df.head()

,"Сумма кредита, руб#",Количество лет проживания в регионе,Социальный статус,Социальный статус супруга(и),Образование,"Стаж работы, лет",Личный доход в месяц после налогооблажения,Наличие личного автомобиля,"Рыночная стоимость автомобиля, руб#",Жилая недвижимость в собственности,"Рыночная стоимость недвижимости, руб#",Наличие кредитов,Возврат кредита
0,47000,15,Работаю/служу,Работает/служит,среднее,8.5,10000,нет,0,Нет,0,нет,0
1,54000,19,Работаю/служу,н/д,начальное,1.0,17000,да,195000,Квартира в многоквартирном доме,1000000,нет,1
2,38000,8,Работаю/служу,н/д,высшее,8.0,10500,да,35000,Квартира в многоквартирном доме,1480000,нет,0
3,25000,19,Работаю/служу,Работает/служит,среднее,1.0,9500,нет,0,Нет,0,нет,0
4,35000,27,Работаю/служу,Работает/служит,неоконченное высшее,2.0,7500,нет,0,Квартира в многоквартирном доме,900000,нет,1


In [6]:
# Дискретизация числовых признаков (квартили или фиксированные границы)
def discretize(s, n_bins=3, labels=None):
    if labels is None:
        labels = [f'Q{i+1}' for i in range(n_bins)]
    # без labels — узнаём реальное число интервалов
    binned = pd.qcut(s, q=n_bins, duplicates='drop')
    n_actual = binned.cat.categories.size
    use_labels = labels[:n_actual] if len(labels) >= n_actual else [f'Q{i+1}' for i in range(n_actual)]
    return pd.qcut(s, q=n_bins, labels=use_labels, duplicates='drop')


num_cols = {
    'Сумма кредита, руб#': ['Малая', 'Средняя', 'Высокая'],
    'Количество лет проживания в регионе': ['Мало', 'Среднее', 'Много'],
    'Стаж работы, лет': ['Мало', 'Среднее', 'Много'],
    'Личный доход в месяц после налогооблажения': ['Низкий', 'Средний', 'Высокий'],
    'Рыночная стоимость автомобиля, руб#': ['Низкая', 'Средняя', 'Высокая'],  # 0 попадает в низ
    'Рыночная стоимость недвижимости, руб#': ['Низкая', 'Средняя', 'Высокая'],
}

for col, lab in num_cols.items():
    if col in df.columns:
        df[col + '_кат'] = discretize(df[col].astype(float), n_bins=len(lab), labels=lab)
            
df.head(10)

,"Сумма кредита, руб#",Количество лет проживания в регионе,Социальный статус,Социальный статус супруга(и),Образование,"Стаж работы, лет",Личный доход в месяц после налогооблажения,Наличие личного автомобиля,"Рыночная стоимость автомобиля, руб#",Жилая недвижимость в собственности,"Рыночная стоимость недвижимости, руб#",Наличие кредитов,Возврат кредита,"Сумма кредита, руб#_кат",Количество лет проживания в регионе_кат,"Стаж работы, лет_кат",Личный доход в месяц после налогооблажения_кат,"Рыночная стоимость автомобиля, руб#_кат","Рыночная стоимость недвижимости, руб#_кат"
0,47000,15,Работаю/служу,Работает/служит,среднее,8.5,10000,нет,0,Нет,0,нет,0,Средняя,Среднее,Много,Низкий,Низкая,Низкая
1,54000,19,Работаю/служу,н/д,начальное,1.0,17000,да,195000,Квартира в многоквартирном доме,1000000,нет,1,Средняя,Много,Мало,Высокий,Средняя,Средняя
2,38000,8,Работаю/служу,н/д,высшее,8.0,10500,да,35000,Квартира в многоквартирном доме,1480000,нет,0,Малая,Среднее,Много,Средний,Низкая,Средняя
3,25000,19,Работаю/служу,Работает/служит,среднее,1.0,9500,нет,0,Нет,0,нет,0,Малая,Много,Мало,Низкий,Низкая,Низкая
4,35000,27,Работаю/служу,Работает/служит,неоконченное высшее,2.0,7500,нет,0,Квартира в многоквартирном доме,900000,нет,1,Малая,Много,Среднее,Низкий,Низкая,Средняя
5,58000,10,Работаю/служу,Работает/служит,среднее,1.0,9000,нет,0,Квартира в многоквартирном доме,1000000,да,1,Высокая,Среднее,Мало,Низкий,Низкая,Средняя
6,44000,6,Работаю/служу,Работает/служит,высшее,1.0,11000,нет,0,Нет,0,нет,1,Средняя,Мало,Мало,Средний,Низкая,Низкая
7,47000,13,Работаю/служу,н/д,среднее,4.0,6000,нет,0,Нет,0,нет,1,Средняя,Среднее,Много,Низкий,Низкая,Низкая
8,32000,6,Работаю/служу,Работает/служит,несколько высших или ученая степень,6.0,8000,да,100000,Квартира в многоквартирном доме,880000,нет,1,Малая,Мало,Много,Низкий,Низкая,Средняя
9,57000,2,Работаю/служу,н/д,высшее,2.0,17000,нет,0,Квартира в многоквартирном доме,120000,нет,1,Высокая,Мало,Среднее,Высокий,Низкая,Низкая


In [7]:
# Собираем все категориальные колонки для транзакций // Не стал брать колонку "Социальный статус" т.к там одно значение
cat_cols = [
    'Социальный статус супруга(и)', 'Образование',
    'Наличие личного автомобиля', 'Жилая недвижимость в собственности', 'Наличие кредитов',
    'Возврат кредита'
]
# Добавляем дискретизированные
cat_cols += [c for c in df.columns if c.endswith('_кат')]

df_cat = df[cat_cols]
df_cat.head()

,Социальный статус супруга(и),Образование,Наличие личного автомобиля,Жилая недвижимость в собственности,Наличие кредитов,Возврат кредита,"Сумма кредита, руб#_кат",Количество лет проживания в регионе_кат,"Стаж работы, лет_кат",Личный доход в месяц после налогооблажения_кат,"Рыночная стоимость автомобиля, руб#_кат","Рыночная стоимость недвижимости, руб#_кат"
0,Работает/служит,среднее,нет,Нет,нет,0,Средняя,Среднее,Много,Низкий,Низкая,Низкая
1,н/д,начальное,да,Квартира в многоквартирном доме,нет,1,Средняя,Много,Мало,Высокий,Средняя,Средняя
2,н/д,высшее,да,Квартира в многоквартирном доме,нет,0,Малая,Среднее,Много,Средний,Низкая,Средняя
3,Работает/служит,среднее,нет,Нет,нет,0,Малая,Много,Мало,Низкий,Низкая,Низкая
4,Работает/служит,неоконченное высшее,нет,Квартира в многоквартирном доме,нет,1,Малая,Много,Среднее,Низкий,Низкая,Средняя


## Применение алгоритма Apriori

### Собственная реализация

In [8]:
from apriori import * 

min_support_val = 0.5
min_confidence_val = 0.6

In [9]:
df = df_cat

print('DataFrame:')
display( df.head())

transactions = dataframe_to_transactions(df, binary=False, prefix_with_column=True)
n_tarn = len(transactions)


print('\nТранзакции:')
for i, t in enumerate(transactions, start=1):
    print(f'T{i}:', t)

frequent_counts = apriori(transactions, min_support=min_support_val)
supports = compute_supports(frequent_counts, n_tarn)

print(f'\nЧастые наборы (support) : {len(supports)} шт.')
for itemset, sup in supports.items():
    print(itemset, '=>', round(sup, 3))

rules = generate_association_rules(frequent_counts, n_tarn, min_confidence=min_confidence_val)
print(f'\nАссоциативные правила (support, confidence): {len(rules)} шт.')
for r in rules:
    print(f'{r.antecedent} -> {r.consequent} (support={r.support:.3f}, conf={r.confidence:.3f})')

DataFrame:


,Социальный статус супруга(и),Образование,Наличие личного автомобиля,Жилая недвижимость в собственности,Наличие кредитов,Возврат кредита,"Сумма кредита, руб#_кат",Количество лет проживания в регионе_кат,"Стаж работы, лет_кат",Личный доход в месяц после налогооблажения_кат,"Рыночная стоимость автомобиля, руб#_кат","Рыночная стоимость недвижимости, руб#_кат"
0,Работает/служит,среднее,нет,Нет,нет,0,Средняя,Среднее,Много,Низкий,Низкая,Низкая
1,н/д,начальное,да,Квартира в многоквартирном доме,нет,1,Средняя,Много,Мало,Высокий,Средняя,Средняя
2,н/д,высшее,да,Квартира в многоквартирном доме,нет,0,Малая,Среднее,Много,Средний,Низкая,Средняя
3,Работает/служит,среднее,нет,Нет,нет,0,Малая,Много,Мало,Низкий,Низкая,Низкая
4,Работает/служит,неоконченное высшее,нет,Квартира в многоквартирном доме,нет,1,Малая,Много,Среднее,Низкий,Низкая,Средняя



Транзакции:
T1: frozenset({'Социальный статус супруга(и) = Работает/служит', 'Количество лет проживания в регионе_кат = Среднее', 'Возврат кредита = 0', 'Рыночная стоимость автомобиля, руб#_кат = Низкая', 'Стаж работы, лет_кат = Много', 'Сумма кредита, руб#_кат = Средняя', 'Рыночная стоимость недвижимости, руб#_кат = Низкая', 'Личный доход в месяц после налогооблажения_кат = Низкий', 'Образование = среднее', 'Наличие личного автомобиля = нет', 'Жилая недвижимость в собственности = Нет', 'Наличие кредитов = нет'})
T2: frozenset({'Количество лет проживания в регионе_кат = Много', 'Возврат кредита = 1', 'Личный доход в месяц после налогооблажения_кат = Высокий', 'Жилая недвижимость в собственности = Квартира в многоквартирном доме', 'Сумма кредита, руб#_кат = Средняя', 'Наличие личного автомобиля = да', 'Образование = начальное', 'Стаж работы, лет_кат = Мало', 'Рыночная стоимость автомобиля, руб#_кат = Средняя', 'Рыночная стоимость недвижимости, руб#_кат = Средняя', 'Наличие кредитов = н

### Проверка с помощью mlxtend

In [10]:
# Преобразование в формат "транзакций" для mlxtend: одна колонка = один предмет (признак=значение)
# Каждая строка — одна анкета (транзакция), колонки — бинарные признаки вида "Образование=высшее"
transactions = []
for _, row in df_cat.iterrows():
    trans = [f"{col} = {val}" for col, val in row.items()]
    transactions.append(trans)

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_transactions = pd.DataFrame(te_ary, columns=te.columns_).astype(bool) 

print('Размер матрицы транзакций:', df_transactions.shape)
df_transactions.head(10)

Размер матрицы транзакций: (83, 34)


,Возврат кредита = 0,Возврат кредита = 1,Жилая недвижимость в собственности = Дом,Жилая недвижимость в собственности = Квартира в многоквартирном доме,Жилая недвижимость в собственности = Нет,Количество лет проживания в регионе_кат = Мало,Количество лет проживания в регионе_кат = Много,Количество лет проживания в регионе_кат = Среднее,Личный доход в месяц после налогооблажения_кат = Высокий,Личный доход в месяц после налогооблажения_кат = Низкий,...,Социальный статус супруга(и) = Не работает,Социальный статус супруга(и) = Работает/служит,Социальный статус супруга(и) = Студент,Социальный статус супруга(и) = н/д,"Стаж работы, лет_кат = Мало","Стаж работы, лет_кат = Много","Стаж работы, лет_кат = Среднее","Сумма кредита, руб#_кат = Высокая","Сумма кредита, руб#_кат = Малая","Сумма кредита, руб#_кат = Средняя"
0,True,False,False,False,True,False,False,True,False,True,...,False,True,False,False,False,True,False,False,False,True
1,False,True,False,True,False,False,True,False,True,False,...,False,False,False,True,True,False,False,False,False,True
2,True,False,False,True,False,False,False,True,False,False,...,False,False,False,True,False,True,False,False,True,False
3,True,False,False,False,True,False,True,False,False,True,...,False,True,False,False,True,False,False,False,True,False
4,False,True,False,True,False,False,True,False,False,True,...,False,True,False,False,False,False,True,False,True,False
5,False,True,False,True,False,False,False,True,False,True,...,False,True,False,False,True,False,False,True,False,False
6,False,True,False,False,True,True,False,False,False,False,...,False,True,False,False,True,False,False,False,False,True
7,False,True,False,False,True,False,False,True,False,True,...,False,False,False,True,False,True,False,False,False,True
8,False,True,False,True,False,True,False,False,False,True,...,False,True,False,False,False,True,False,False,True,False
9,False,True,False,True,False,True,False,False,True,False,...,False,False,False,True,False,False,True,True,False,False


In [11]:
frequent_itemsets_mlxtend = mlxtend_apriori(df_transactions, min_support=min_support_val, use_colnames=True)

print(f'Частые наборы (mlxtend, min_support={min_support_val}): {len(frequent_itemsets_mlxtend)} шт.')
for index, row in frequent_itemsets_mlxtend.sort_values(by='support').iterrows():
    print(row['itemsets'], '=>', round(row['support'], 3))

rules_mlxtend = mlxtend_association_rules(frequent_itemsets_mlxtend, metric="confidence", min_threshold=min_confidence_val)

print(f'\nАссоциативные правила (mlxtend, min_confidence={min_confidence_val}): {len(rules_mlxtend)} шт.')
for index, r in rules_mlxtend.iterrows():
    print(f'{r.antecedents} -> {r.consequents} (support={r.support:.3f}, conf={r.confidence:.3f})')

Частые наборы (mlxtend, min_support=0.5): 15 шт.
frozenset({'Рыночная стоимость автомобиля, руб#_кат = Низкая', 'Возврат кредита = 1'}) => 0.518
frozenset({'Жилая недвижимость в собственности = Нет', 'Наличие кредитов = нет'}) => 0.53
frozenset({'Жилая недвижимость в собственности = Нет', 'Рыночная стоимость недвижимости, руб#_кат = Низкая', 'Наличие кредитов = нет'}) => 0.53
frozenset({'Наличие личного автомобиля = нет'}) => 0.542
frozenset({'Возврат кредита = 1', 'Рыночная стоимость недвижимости, руб#_кат = Низкая'}) => 0.542
frozenset({'Рыночная стоимость автомобиля, руб#_кат = Низкая', 'Наличие личного автомобиля = нет'}) => 0.542
frozenset({'Рыночная стоимость автомобиля, руб#_кат = Низкая', 'Наличие кредитов = нет'}) => 0.578
frozenset({'Рыночная стоимость недвижимости, руб#_кат = Низкая', 'Наличие кредитов = нет'}) => 0.59
frozenset({'Жилая недвижимость в собственности = Нет', 'Рыночная стоимость недвижимости, руб#_кат = Низкая'}) => 0.602
frozenset({'Жилая недвижимость в собств

In [12]:
# Сравнение результатов собственной реализации и mlxtend

rules_custom_set = {(r.antecedent, r.consequent): (r.support, r.confidence) for r in rules}

rules_mlxtend_set = {}
for _, row in rules_mlxtend.iterrows():
    ant = row['antecedents']
    cons = row['consequents']
    sup = row['support']
    conf = row['confidence']
    rules_mlxtend_set[(ant, cons)] = (sup, conf)

common = set(rules_custom_set) & set(rules_mlxtend_set)
only_custom = set(rules_custom_set) - set(rules_mlxtend_set)
only_mlxtend = set(rules_mlxtend_set) - set(rules_custom_set)

print('Сравнение ассоциативных правил:')
print(f'  Собственная реализация: {len(rules)} правил')
print(f'  mlxtend:                {len(rules_mlxtend)} правил')
print(f'  Совпадают:              {len(common)} правил')
if only_custom:
    print(f'  Только в собственной:   {len(only_custom)}')
if only_mlxtend:
    print(f'  Только в mlxtend:       {len(only_mlxtend)}')
if len(common) == len(rules) == len(rules_mlxtend) and not only_custom and not only_mlxtend:
    print('\nРезультаты полностью совпадают. Собственная реализация Apriori корректна.')

Сравнение ассоциативных правил:
  Собственная реализация: 22 правил
  mlxtend:                22 правил
  Совпадают:              22 правил

Результаты полностью совпадают. Собственная реализация Apriori корректна.


## Анализ результатов поиска ассоциативных правил

Прежде чем интерпретировать правила, зафиксируем опорные точки. Целевая переменная **«Возврат кредита = 1»** имеет support = 0.759 — это означает, что ~76% наблюдений в выборке относятся к классу «кредит возвращён».

### Анализ частых наборов

Наблюдается явный кластер взаимосвязанных признаков вокруг «бедности активов»:
* «Наличие кредитов = нет» (0.88) — доминирующий признак, встречается в 88% записей
* «Возврат кредита = 1» (0.759)
* «Рыночная стоимость недвижимости = Низкая» (0.663) и «Наличие кредитов = нет + Возврат кредита = 1» (0.663)

Все 3-элементные наборы замкнуты вокруг одной тройки (с support = 0.53): 
* Низкая стоимость недвижимости, 
* Нет кредитов, 
* Нет жилья в собственности

Это говорит о том, что данные признаки образуют один плотный кластер «клиентов без активов», а не несколько независимых паттернов.

### Анализ ассоциативных правил

#### Правила с confidence = 1.0 (детерминированные)
```
{Жилья в собственности = Нет} → {Стоимость недвижимости = Низкая}  (sup=0.602, conf=1.000)
{Нет автомобиля} → {Стоимость автомобиля = Низкая}                 (sup=0.542, conf=1.000)
```

* Эти правила являются прямым следствием логики кодирования данных. Если у клиента нет имущества, его рыночная стоимость категоризируется как «Низкая» (фактически — нулевая). 
* Это артефакт feature engineering, а не содержательный паттерн. 
* В продуктивной модели такие признаки избыточны и должны быть консолидированы.

#### Ключевые правила для бизнес-задачи

Наиболее интересны правила с участием целевой переменной:

| Правило | Support | Confidence | Lift\* |
| --- | --- | --- | --- |
| {Нет кредитов} → {Возврат = 1} | 0.663 | 0.753 | ~0.99 |
| {Стоимость недвижимости = Низкая} → {Возврат = 1} | 0.542 | 0.818 | ~1.08 |
| {Стоимость автомобиля = Низкая} → {Возврат = 1} | 0.518 | 0.768 | ~1.01 |
| {Возврат = 1} → {Нет кредитов} | 0.663 | 0.873 | ~0.99 |

`Lift = conf / P(consequent). P(Возврат=1) ≈ 0.759, P(Нет кредитов) ≈ 0.880`

lift для большинства правил с «Возврат кредита = 1» близок к 1.0, что означает — предпосылки практически не улучшают предсказание относительно базовой частоты класса.

### Вывод

Полученные правила подтверждают интуитивно тривиальный профиль: клиенты без значимых активов и без кредитной истории в большинстве своём возвращают небольшие кредиты. Для построения скоринговой модели данный набор правил в текущем виде имеет ограниченную практическую ценность и требует доработки признакового пространства.